# سیستم RAG برای استخراج خواص فیزیکی مواد از مقالات

**نسخه‌ی نوت‌بوک: 13** (اگر عدد کمتری می‌بینید، نوت‌بوک را دوباره از لینک باز کنید)

**قبل از شروع:** منوی *Runtime → Change runtime type* → **T4 GPU**

سلول‌ها را **به ترتیب** اجرا کنید (دکمه‌ی ▶):

| سلول | کار | چند بار؟ |
|---|---|---|
| ۱ تا ۳ | آماده‌سازی | هر بار |
| ۴ | مرتب کردن PDFهای پوشه‌ی `RAG-Optics` در پوشه‌ی هر بلور و گزارش وضعیت | هر بار (سریع؛ فایلی پاک نمی‌شود) |
| ۵ | پردازش مقاله‌ها و ساخت پایگاه داده | بار اول طولانی (چند ساعت)؛ بعد فقط مقاله‌های تازه |
| ۶ و ۷ | مدل زبانی و باز کردن برنامه | هر بار |

**سریع‌تر کردن سلول ۵:** نوت‌بوک‌های «کارگر» (`colab_worker_1` تا `colab_worker_4`) را هم‌زمان در
تب‌های دیگر اجرا کنید؛ هر کدام بخشی از PDFها را تبدیل می‌کند و نتیجه را در همان پوشه‌ی Drive می‌گذارد.
سلول ۵ فقط کارهای باقی‌مانده را انجام می‌دهد. نیازی نیست فایلی برای من بفرستید.

همه‌چیز در **Google Drive** ذخیره می‌شود و با بستن کولب پاک نمی‌شود؛ اگر وسط کار قطع شد، دوباره اجرا کنید
تا از همان‌جا ادامه دهد.

In [ ]:
#@title ۱. بررسی کارت گرافیک
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "⚠️ کارت گرافیک فعال نیست: Runtime → Change runtime type → T4 GPU"

In [ ]:
#@title ۲. اتصال به Google Drive

import os
# Keep Colab's preinstalled JAX/TensorFlow from reserving the GPU memory
# (otherwise the embedding model and Ollama run out of memory).
os.environ.update({'JAX_PLATFORMS': 'cpu', 'XLA_PYTHON_CLIENT_PREALLOCATE': 'false',
                   'TF_FORCE_GPU_ALLOW_GROWTH': 'true', 'USE_TF': '0', 'USE_FLAX': '0',
                   'TF_CPP_MIN_LOG_LEVEL': '3'})
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/matrag_data'
os.makedirs(f'{WORK}/logs', exist_ok=True)
os.environ['MATRAG_DATA_DIR'] = f'{WORK}/data'
# The vector database runs on Colab's local disk (SQLite is unreliable on Drive);
# the app copies it back to Drive after every change.
os.environ['MATRAG_STORAGE_DIR'] = '/content/storage'
os.environ['MATRAG_STORAGE_BACKUP_DIR'] = f'{WORK}/storage'
# refractiveindex.info database (downloaded in cell 3), shared by all corpora. It lives on
# Colab's local disk: reading its thousands of small files from Drive takes many minutes.
os.environ['MATRAG_REFERENCE_DIR'] = '/content/refractiveindex/database'
!mkdir -p "{WORK}/storage" /content/storage && cp -rT "{WORK}/storage" /content/storage
print('✅ Data folder:', WORK)

In [ ]:
#@title ۳. نصب برنامه از GitHub

BRANCH = "claude/festive-bardeen-nnw123"  #@param {type:"string"}
# Leave the project folder before deleting it (re-running this cell used to fail
# with "Unable to read current working directory").
%cd /content
!rm -rf /content/RAG && git clone -q --depth 1 -b {BRANCH} https://github.com/aminsadidi/RAG.git /content/RAG
%cd /content/RAG
!pip install -q -e . 2>&1 | grep -iE "^error" || true
# An editable install is only seen by Python after a restart; adding the source
# folder to the path makes it importable in this session right away.
import sys
if '/content/RAG/src' not in sys.path:
    sys.path.insert(0, '/content/RAG/src')

# refractiveindex.info database (CC0), used to check formulas extracted from optical papers;
# downloaded to the local disk in each session (a few seconds).
if not os.path.isdir('/content/refractiveindex/database'):
    !git clone -q --depth 1 https://github.com/polyanskiy/refractiveindex.info-database.git /content/refractiveindex && echo "refractiveindex.info database downloaded"
!git log --oneline -1
!matrag --help > /dev/null && echo '✅ Installed (red "dependency conflict" warnings from pip can be ignored)' || echo '❌ Installation failed: run this cell again and send the output'

In [ ]:
#@title ۴. پوشه‌ی مقاله‌ها: مرتب‌سازی و وضعیت
#@markdown مسیر پوشه‌ی مقاله‌ها در Drive:
COLLECTION = "/content/drive/MyDrive/RAG-Optics"  #@param {type:"string"}
#@markdown **ORGANIZE**: هر PDF (مثلاً فایل‌های `05_Downloaded_PDFs`) به پوشه‌ی دسته و بلور خودش در
#@markdown `01_Papers_by_Category` منتقل می‌شود. هیچ فایلی پاک نمی‌شود؛ PDFهایی که در فهرست اصلی نیستند به
#@markdown `99_Not_in_master_index` و نسخه‌های تکراری به `98_Duplicate_copies` می‌روند.
#@markdown گزارش کامل: `02_Master_Index/pdf_inventory.csv`
ORGANIZE = True  #@param {type:"boolean"}
os.environ['MATRAG_CORPUS'] = 'rag-optics'
print('1/3 Linking the folder...')
!matrag --corpus rag-optics link "{COLLECTION}" 2>&1 | grep -vE "Warning"
if ORGANIZE:
    print('2/3 Sorting the PDFs (a few minutes the first time)...')
    !matrag --corpus rag-optics organize --move 2>&1 | grep -vE "Warning"
print('3/3 Status:')
!matrag --corpus rag-optics status 2>&1 | grep -vE "Warning"

In [ ]:
#@title ۵. پردازش مقاله‌ها و ساخت پایگاه داده
#@markdown بار اول برای حدود ۲۰۰۰ PDF چند ساعت طول می‌کشد (تبدیل PDF کُندترین مرحله است). اگر
#@markdown نوت‌بوک‌های کارگر را هم‌زمان اجرا کرده‌اید، این سلول را **بعد از تمام شدن آن‌ها** اجرا کنید
#@markdown (یا همین حالا؛ فقط ممکن است چند مقاله دو بار تبدیل شود). مقاله‌های بدون PDF با چکیده‌شان وارد
#@markdown می‌شوند. اگر کولب قطع شد، دوباره اجرا کنید: کار از همان‌جا ادامه پیدا می‌کند.
%cd {WORK}
!matrag --corpus rag-optics ingest --prune 2>&1 | grep -vE "Warning|Loading|it/s\]" | tee -a "{WORK}/logs/ingest.log"
%cd /content/RAG
!matrag --corpus rag-optics status 2>&1 | grep -vE "Warning"

In [ ]:
#@title ۶. راه‌اندازی مدل زبانی
#@markdown **ollama**: مدل متن‌باز روی کارت گرافیک کولب (رایگان، بدون سهمیه). بار اول دانلود مدل چند دقیقه طول می‌کشد.
#@markdown **gemini**: کلید API را در بخش Secrets کولب (آیکون 🔑 سمت چپ) با نام `MATRAG_GOOGLE_API_KEY` ذخیره کنید.
LLM_PROVIDER = "ollama"  #@param ["ollama", "gemini"]
#@markdown qwen3:8b کنار برنامه در حافظه‌ی کارت گرافیک T4 جا می‌شود؛ 14b ممکن است جا نشود و خیلی کُند شود.
OLLAMA_MODEL = "qwen3:8b"  #@param ["qwen3:8b", "qwen3:14b", "qwen3:4b-instruct-2507-q4_K_M"]
#@markdown **KEEP_MODEL_ON_DRIVE**: مدل یک بار در Drive ذخیره شود (حدود ۵ گیگ برای 8b) تا دفعه‌های بعد دانلود نشود.
KEEP_MODEL_ON_DRIVE = False  #@param {type:"boolean"}
import os, subprocess, time
os.environ['MATRAG_LLM_PROVIDER'] = LLM_PROVIDER
os.environ['MATRAG_OLLAMA_MODEL'] = OLLAMA_MODEL
os.environ['MATRAG_OLLAMA_THINKING'] = 'false'  # faster answers from Qwen3

try:  # Gemini is optional; with a key both models can be chosen in the app
    from google.colab import userdata
    os.environ['MATRAG_GOOGLE_API_KEY'] = userdata.get('MATRAG_GOOGLE_API_KEY')
    print('✅ Gemini key loaded')
except Exception:
    print('ℹ️ No Gemini key in Colab Secrets (only needed for gemini)')

if LLM_PROVIDER == "ollama":
    os.environ['OLLAMA_MODELS'] = '/content/ollama_models'
    DRIVE_MODELS = f'{WORK}/ollama_models'
    if KEEP_MODEL_ON_DRIVE and os.path.isdir(DRIVE_MODELS):
        print('Copying saved model from Drive...')
        !mkdir -p /content/ollama_models && cp -rT "{DRIVE_MODELS}" /content/ollama_models
    if subprocess.run('which ollama', shell=True, capture_output=True).returncode != 0:
        !apt-get install -y -qq zstd > /dev/null
        !curl -fsSL https://ollama.com/install.sh | sh > /dev/null
    subprocess.Popen('ollama serve > /content/ollama.log 2>&1', shell=True)
    time.sleep(5)
    !ollama pull {OLLAMA_MODEL} 2>&1 | tail -1
    if KEEP_MODEL_ON_DRIVE:
        !mkdir -p "{DRIVE_MODELS}" && cp -rTu /content/ollama_models "{DRIVE_MODELS}" && echo "Model saved to Drive."
print('✅ Language model ready:', LLM_PROVIDER)

In [ ]:
#@title ۷. اجرای برنامه 🚀
#@markdown مجموعه‌ی پیش‌فرض (در خود برنامه هم قابل تغییر است). `rag-optics` همان پوشه‌ی مقاله‌های Drive است.
CORPUS = "rag-optics"  #@param ["rag-optics", "optical", "hitran"] {allow-input: true}
#@markdown **PUBLIC_LINK** (پیشنهادی): برنامه با یک لینک موقت https://….gradio.live باز می‌شود — روی لینکی که
#@markdown پایین این سلول چاپ می‌شود بزنید. لینک خصوصی نیست (هر کس آن را داشته باشد باز می‌کند) و با بسته شدن
#@markdown کولب از کار می‌افتد. خاموش: نمایش داخل کولب (ممکن است ظاهر برنامه درست بارگذاری نشود).
PUBLIC_LINK = True  #@param {type:"boolean"}
import os, sys
os.environ['MATRAG_CORPUS'] = CORPUS
if '/content/RAG/src' not in sys.path:  # see cell 3
    sys.path.insert(0, '/content/RAG/src')
# Forget code imported by an earlier run, so the version just installed in cell 3 is used.
for name in [m for m in sys.modules if m == 'matrag' or m.startswith('matrag.')]:
    del sys.modules[name]
from matrag.webapp import launch
PORT = launch(port=7860, share=PUBLIC_LINK, inline=False, prevent_thread_lock=True, quiet=not PUBLIC_LINK)
if not PUBLIC_LINK:
    from google.colab import output
    output.serve_kernel_port_as_window(PORT, anchor_text='👉 باز کردن برنامه در یک تب جدید (پیشنهاد می‌شود)')
    output.serve_kernel_port_as_iframe(PORT, height=950)

---
### (اختیاری) ارزیابی

این سلول‌ها جدول‌های نتایج مقاله را می‌سازند. فایل‌ها در `matrag_data/results/eval` ذخیره می‌شوند.
سؤال‌های مرجع در `matrag_data/data/<corpus>/gold/questions.csv` قرار دارند.

In [ ]:
#@title ۸. ارزیابی بازیابی (بدون مصرف مدل زبانی)
#@markdown برای `rag-optics` سؤال‌ها خودکار از refractiveindex.info ساخته می‌شوند: «فرمول پاشندگی ماده‌ی X»
#@markdown باید یکی از مقاله‌هایی را پیدا کند که پایگاه برای آن ماده به آن ارجاع داده است.
CORPUS = globals().get('CORPUS', 'rag-optics')  # set by cell 7; rag-optics if it was skipped
%cd {WORK}
if CORPUS == 'rag-optics':
    !matrag --corpus rag-optics evaluate build-gold 2>&1 | grep -vE "Warning|Loading"
else:
    !mkdir -p "{WORK}/data/{CORPUS}/gold" && cp -n /content/RAG/data/{CORPUS}/gold/*.csv "{WORK}/data/{CORPUS}/gold/" 2>/dev/null; true
!matrag --corpus {CORPUS} evaluate retrieval 2>&1 | grep -vE "Warning|Loading" | tee -a "{WORK}/logs/eval.log"
%cd /content/RAG

In [ ]:
#@title ۹. ارزیابی پاسخ‌ها: RAG در برابر مدل بدون منبع (برای hitran و optical)
#@markdown سؤال‌های خودکارِ `rag-optics` جواب مرجع ندارند و این‌جا رد می‌شوند؛ ارزیابی اصلی آن سلول ۱۰ است.
LIMIT = 16  #@param {type:"integer"}
CORPUS = globals().get('CORPUS', 'rag-optics')  # set by cell 7
%cd {WORK}
!matrag --corpus {CORPUS} evaluate answers --limit {LIMIT} 2>&1 | grep -vE "Warning|Loading" | tee -a "{WORK}/logs/eval.log"
%cd /content/RAG

In [ ]:
#@title ۱۰. ارزیابی اصلی: استخراج فرمول ← مقایسه با refractiveindex.info
#@markdown برای هر مقاله‌ای که refractiveindex.info فرمولش را از آن برداشته: فرمول پاشندگی از متن مقاله استخراج و
#@markdown ضریب شکست حاصل با پایگاه مقایسه می‌شود (بیشینه‌ی |Δn|). به مدل زبانی (سلول ۶) نیاز دارد و برای هر مقاله
#@markdown چند دقیقه طول می‌کشد؛ هر بار **LIMIT** مقاله‌ی تازه پردازش می‌شود و اجرای بعدی از همان‌جا ادامه می‌دهد.
LIMIT = 20  #@param {type:"integer"}
%cd {WORK}
!matrag --corpus rag-optics evaluate formulas --limit {LIMIT} 2>&1 | grep -vE "Warning|Loading|it/s\]" | tee -a "{WORK}/logs/eval.log"
%cd /content/RAG

---
### انتقال به سایت

سایت (Cloudflare) از روی یک کپی از پایگاه داده در Qdrant Cloud جست‌وجو می‌کند. بعد از سلول ۵ (و هر بار که
مقاله‌ی تازه اضافه شد) این سلول را اجرا کنید. به کارت گرافیک و مدل زبانی نیاز ندارد.

In [ ]:
#@title ۱۱. بارگذاری پایگاه داده در سایت (Qdrant Cloud)
#@markdown کلید Qdrant خودکار از فایل `matrag_data/qdrant.env` در Drive خوانده می‌شود (یا از Secrets کولب).
#@markdown بردارها دوباره ساخته نمی‌شوند؛ همان بردارهای پایگاه (و ارزیابی‌ها) فرستاده می‌شوند. اگر قطع شد دوباره
#@markdown اجرا کنید: بخش‌های فرستاده‌شده رد می‌شوند. **RECREATE** مجموعه را پاک و از نو می‌سازد.
RECREATE = False  #@param {type:"boolean"}
#@markdown **EVALUATE**: بعد از بارگذاری، ارزیابی سلول ۸ روی جست‌وجوی سایت هم اجرا شود (برای مقایسه).
EVALUATE = True  #@param {type:"boolean"}

# Keys: matrag_data/qdrant.env (Qdrant) and matrag_data/cloudflare.env (the site's PDFs) on Drive,
# as KEY=value lines; the Qdrant ones can also come from Colab Secrets.
import os
KEYS_FILE = f'{WORK}/qdrant.env'
for keys_file in (KEYS_FILE, f'{WORK}/cloudflare.env'):
    if os.path.exists(keys_file):
        for line in open(keys_file, encoding='utf-8'):
            key, sep, value = line.strip().partition('=')
            if sep and not key.startswith('#'):
                os.environ[key.strip()] = value.strip().strip('"')
if not os.environ.get('QDRANT_API_KEY'):
    from google.colab import userdata
    for key in ('QDRANT_URL', 'QDRANT_API_KEY'):
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
assert os.environ.get('QDRANT_URL') and os.environ.get('QDRANT_API_KEY'), f'Qdrant key not found ({KEYS_FILE})'

%cd {WORK}
!matrag --corpus rag-optics export-qdrant {'--recreate' if RECREATE else ''} 2>&1 | grep -vE "Warning|Loading"
if EVALUATE and os.path.exists(f'{WORK}/data/rag-optics/gold/questions.csv'):
    !matrag --corpus rag-optics evaluate retrieval --backend qdrant 2>&1 | grep -vE "Warning|Loading" | tee -a "{WORK}/logs/eval.log"
%cd /content/RAG